In [2]:
### Import Tracer
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

Overriding of current TracerProvider is not allowed


In [3]:
with tracer.start_as_current_span("say_hello_loop") as span:
    # your code here
    for i in range(10):
        print(f"hello world!{i}")
    span.set_attribute("final_i", i)

hello world!0
hello world!1
hello world!2
hello world!3
hello world!4
hello world!5
hello world!6
hello world!7
hello world!8
hello world!9
{
    "name": "say_hello_loop",
    "context": {
        "trace_id": "0xc77b2f73e5d18272e1d4598eda113b84",
        "span_id": "0x49280b5097dc628e",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-23T00:54:24.495136Z",
    "end_time": "2026-07-23T00:54:24.495305Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "final_i": 9
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "603ab4e0-7bbc-4c02-98d0-6f9d12b4cfbb",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


In [4]:
with tracer.start_as_current_span("span_test2") as span:
    print("testing a new span -span2")
    span.set_attribute("final_i", 2)

testing a new span -span2
{
    "name": "span_test2",
    "context": {
        "trace_id": "0x9858154885c6d4060009c861a9a90c0f",
        "span_id": "0x4331cd652a0617f8",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-23T00:54:28.869951Z",
    "end_time": "2026-07-23T00:54:28.870090Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "final_i": 2
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "603ab4e0-7bbc-4c02-98d0-6f9d12b4cfbb",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}



## Q1. First trace

Wrap the `rag()` method so each call produces a span. The simplest way
is to create a `RAGTraced` subclass of `RAGBase` that wraps `rag()`,
`search()`, and `llm()` each in their own span.

Run this query:

> How does the agentic loop keep calling the model until it stops?

The console exporter prints every finished span as a dictionary.
Count the spans in the console output - each one is a separate
`ReadableSpan` entry. How many spans does the trace produce?

* 1
* 3
* 5
* 7

ANS:3

In [4]:
from openai import OpenAI

from gitsource import GithubRepositoryDataReader
from minsearch import Index

from rag_helper import RAGBase

from dotenv import load_dotenv
load_dotenv()

COMMIT = "8c1834d"

# --- Load the course lessons (same as HW1, HW2, HW4) ---
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

client = OpenAI()


class RAGTraced(RAGBase):

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            # your code here
            result = super().rag(query)
            span.set_attribute("query", query)
            return result
        
    
    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            # your code here
            result = super().search(query, num_results)
            span.set_attribute("query", query)
            span.set_attribute("num_results", num_results)
            return result
       

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            # your code here
            result = super().llm(prompt)
            #span.set_attribute("prompt", prompt)
            span.set_attribute("input_tokens", result.usage.input_tokens)
            span.set_attribute("output_tokens", result.usage.output_tokens)
            return result

In [14]:
rag1 = RAGTraced(index=index, llm_client=client)
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag1.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x8e88c778b455da33a5de857e40fec59b",
        "span_id": "0x4a4a187b36782cb2",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x1f91b651f1d88d2f",
    "start_time": "2026-07-23T01:05:38.976429Z",
    "end_time": "2026-07-23T01:05:38.978972Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "603ab4e0-7bbc-4c02-98d0-6f9d12b4cfbb",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x8e88c778b455da33a


## Q2. Capturing metrics as span attributes

Spans are not just timing markers - you can attach any information you
want to them with `set_attribute`. We already use spans to record how
long each step takes. Now we'll add the metrics we care about: tokens
and cost.

Read the token usage from the LLM response (the `llm()` method in the
starter already returns the raw response object) and set them as
attributes on the `llm` span:

```python
span.set_attribute("input_tokens", usage.input_tokens)
span.set_attribute("output_tokens", usage.output_tokens)
```

And since we know both input and output tokens, we can also compute
the cost using the code from the previous modules.

Now re-run the query. How many input tokens do we see?

* 700
* 7000
* 70000
* 700000

> These numbers vary between runs. Pick the closest option.

ANS: 7000

In [19]:
rag1 = RAGTraced(index=index, llm_client=client)
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag1.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x6f1e61e80464601d5b71d6122ed14f92",
        "span_id": "0x7de01a2015424a78",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xdeebcbcc4777ff2f",
    "start_time": "2026-07-23T01:20:56.523686Z",
    "end_time": "2026-07-23T01:20:56.525265Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "603ab4e0-7bbc-4c02-98d0-6f9d12b4cfbb",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x6f1e61e80464601d5


## Q3. Span timing

Each span automatically records its duration. Look at the console output
from Q1 and find the durations for the `search` span and the `llm` span.

For a typical query, roughly how long does the LLM call take?

* Under 100ms
* 100-500ms
* 500-2000ms
* Over 2000ms

> The first call can be slower (cold start). Pick the range you see
> most often.

ANS: Over 2000ms


## Q4. Saving traces to SQLite

Right now the spans are printed to the terminal and then gone. We don't
save them.

We want to persist them so we can query them later.

In this homework, we'll use SQLite - it's a more lightweight option than
Postgres, so we don't need to set up any docker containers in this homework.

Our instrumentation is already done, we don't need to change anything there.
But we need to create a custom exporter. Instead of printing the spans,
it will save them to the database.

OTel calls the exporter through the same span processor we already use,
we just swap the destination.

Now we will create a custom exporter that saves each finished span to a
SQLite database. The exporter extends `SpanExporter`. It has the following methods:

- `export` method that receives a list of `ReadableSpan` objects
- `shutdown` and `force_flush` methods

Let's implement it:

```python
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True
```

Replace the console exporter with this new exporter:

```python
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
```

Re-run the query from Q1. Which span names appear in the `spans` table?

* Only `rag`
* `rag` and `llm`
* `rag`, `search`, and `llm`
* `search`, `llm`, and `judge`

ANS:`rag`, `search`, and `llm`

In [2]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [3]:
### Import Tracer
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")



In [6]:
rag1 = RAGTraced(index=index, llm_client=client)
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag1.rag(query)
print(answer)

The loop keeps calling the model by checking whether the latest response contains any `function_call` items.

- It sends the current `messages` history to the model.
- If the model returns a function call, the code runs the tool, appends the tool output to `messages`, and sets `has_function_calls = True`.
- Then the `while True` loop repeats.
- If the model returns only a normal `message` and no function calls, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is: **no function calls in the response**.


# Q5. Querying trace data

The traces are now in SQLite. Run one more query through the traced
RAG, then query the database.

The `rag` span wraps everything, so its duration includes both
`search` and `llm`. To see where time actually goes, exclude the
`rag` span and compare the children.

Using SQL (or pandas), compute the total duration for each span name
excluding `rag`. Which span type takes the most total time?

* `search`
* `llm`
* They're all about the same

ANS: llm

In [20]:
import sqlite3
import pandas as pd

# Establish connection to the local database file
conn = sqlite3.connect("traces.db")

# Run query and immediately populate your DataFrame
query = "SELECT * FROM spans"
df = pd.read_sql_query(query, conn)

# Close resource to free lock files
conn.close()

print(df.head())

     name           start_time             end_time  input_tokens  \
0  search  1784854048750832648  1784854048803344321           NaN   
1     llm  1784854048812329603  1784854052379388837        7111.0   
2     rag  1784854048750774830  1784854052385256347           NaN   
3  search  1784854311135022218  1784854311137464096           NaN   
4     llm  1784854311144344654  1784854313310453937        7111.0   

   output_tokens  cost  
0            NaN  None  
1          135.0  None  
2            NaN  None  
3            NaN  None  
4          125.0  None  


In [23]:
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784854048750832648,1784854048803344321,NaN,NaN,None
1,llm,1784854048812329603,1784854052379388837,7111.0,135.0,None
2,rag,1784854048750774830,1784854052385256347,NaN,NaN,None
3,search,1784854311135022218,1784854311137464096,NaN,NaN,None
4,llm,1784854311144344654,1784854313310453937,7111.0,125.0,None
5,rag,1784854311134984794,1784854313316615578,NaN,NaN,None


In [24]:
df['duration_in_ms'] = (df['end_time'] - df['start_time']) / 1e6


In [26]:
df[df.name!='rag']

,name,start_time,end_time,input_tokens,output_tokens,cost,duration_in_ms
0,search,1784854048750832648,1784854048803344321,NaN,NaN,None,52.511673
1,llm,1784854048812329603,1784854052379388837,7111.0,135.0,None,3567.059234
3,search,1784854311135022218,1784854311137464096,NaN,NaN,None,2.441878
4,llm,1784854311144344654,1784854313310453937,7111.0,125.0,None,2166.109283


# Q6. Token stability across runs

Load the SQLite data with pandas. One thing a dashboard can tell you
is how stable your system is. If the same query always produces the
same number of input tokens, the context your RAG retrieves is
consistent. If it varies a lot, something in the search may be
unstable.

Run the same query from Q1 three more times (so you have 4 RAG calls
total in the database). Then compute the input tokens for each `llm`
span.

How much do the input tokens vary across these 4 runs?

* They're identical
* Within 10% of each other
* Within 50% of each other
* They vary more than 50%

ANS:They're identical

In [27]:
rag1 = RAGTraced(index=index, llm_client=client)
query = "How does the agentic loop keep calling the model until it stops?"
## run the query 3 more times
for i in range(3):
    answer = rag1.rag(query)
    print(answer)

It keeps calling the model in a `while True` loop.

After each model response, the code checks whether there were any `function_call` items. If there were, it runs the tool, appends the tool result to `messages`, and loops again. If there are no function calls, it breaks.

So the stop condition is:

- **function call present** → keep looping
- **no function calls** → stop and return the final answer

If you want, I can also show the exact loop logic in plain English step by step.
The loop keeps calling the model inside a `while True` loop and checks whether the response includes any `function_call` items.

- If the model returns a function call, the code runs the tool, appends the tool result to `messages`, and continues looping.
- If the model returns only a normal message and no function calls, `has_function_calls` stays `False`, and the loop breaks.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In short: it keeps calling the model until the mod

In [28]:
import sqlite3
import pandas as pd

# Establish connection to the local database file
conn = sqlite3.connect("traces.db")

# Run query and immediately populate your DataFrame
query = "SELECT * FROM spans"
df = pd.read_sql_query(query, conn)

# Close resource to free lock files
conn.close()

print(df.head())

     name           start_time             end_time  input_tokens  \
0  search  1784854048750832648  1784854048803344321           NaN   
1     llm  1784854048812329603  1784854052379388837        7111.0   
2     rag  1784854048750774830  1784854052385256347           NaN   
3  search  1784854311135022218  1784854311137464096           NaN   
4     llm  1784854311144344654  1784854313310453937        7111.0   

   output_tokens  cost  
0            NaN  None  
1          135.0  None  
2            NaN  None  
3            NaN  None  
4          125.0  None  


In [29]:
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784854048750832648,1784854048803344321,NaN,NaN,None
1,llm,1784854048812329603,1784854052379388837,7111.0,135.0,None
2,rag,1784854048750774830,1784854052385256347,NaN,NaN,None
3,search,1784854311135022218,1784854311137464096,NaN,NaN,None
4,llm,1784854311144344654,1784854313310453937,7111.0,125.0,None
5,rag,1784854311134984794,1784854313316615578,NaN,NaN,None
6,search,1784855271129642063,1784855271131214383,NaN,NaN,None
7,llm,1784855271137482455,1784855277387465524,7111.0,116.0,None
8,rag,1784855271129617653,1784855277394525360,NaN,NaN,None
9,search,1784855277400447290,1784855277401955453,NaN,NaN,None
